## Section 1: Imports
Load all required libraries — PyTorch, Transformers (DistilBERT), sklearn, NLTK, plotting, etc.

In [1]:
# Import all required libraries for data processing, deep learning, and visualization
import os, re, ssl, warnings, json
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')          # non-interactive backend so plots save to files
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
ssl._create_default_https_context = ssl._create_unverified_context   # fix SSL for NLTK downloads
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.stem   import WordNetLemmatizer

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim       import AdamW

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.model_selection   import train_test_split
from sklearn.metrics           import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from collections import Counter

warnings.filterwarnings('ignore')
print("All imports successful ✅")


All imports successful ✅


## Section 2: Configuration
Set file paths, model hyper-parameters, label mapping, and device (GPU / CPU).

In [2]:
# --- Paths ---
# When running inside a notebook the working dir is the notebooks/ folder,
# so we go one level up to reach the project root.
BASE_DIR   = os.path.dirname(os.path.abspath(''))   # project root
DATA_DIR   = os.path.join(BASE_DIR, 'Ecommerce_dataset')
OUTPUT_DIR = os.path.join(BASE_DIR, 'personal_update', 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH       = os.path.join(DATA_DIR, 'train_data.csv')
TEST_PATH        = os.path.join(DATA_DIR, 'test_data.csv')
TEST_HIDDEN_PATH = os.path.join(DATA_DIR, 'test_data_hidden.csv')

# --- Model hyper-parameters ---
MODEL_NAME       = 'distilbert-base-uncased'   # lighter, faster BERT variant
MAX_LENGTH       = 128                          # max tokens per review
BATCH_SIZE       = 16
EPOCHS           = 4
LEARNING_RATE    = 2e-5
VALIDATION_SPLIT = 0.15                         # 15 % held out for validation
RANDOM_SEED      = 42

# --- Label mapping ---
LABEL_MAP   = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
LABEL_NAMES = ['Negative', 'Neutral', 'Positive']

# --- Device ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")


Using device: cpu


## Section 3: Data Loading
Read the three CSV files (train, test, test_hidden) and inspect shapes, sentiment distribution, and missing values.

In [3]:
# Load the three datasets
train_df       = pd.read_csv(TRAIN_PATH)
test_df        = pd.read_csv(TEST_PATH)
test_hidden_df = pd.read_csv(TEST_HIDDEN_PATH)

print(f"Training data:    {train_df.shape[0]} rows, {train_df.shape[1]} columns")
print(f"Test data:        {test_df.shape[0]} rows, {test_df.shape[1]} columns")
print(f"Test hidden data: {test_hidden_df.shape[0]} rows, {test_hidden_df.shape[1]} columns")

print(f"\nTraining sentiment distribution:")
print(train_df['sentiment'].value_counts())
print(f"\nMissing values in training data:")
print(train_df.isnull().sum())


Training data:    4000 rows, 8 columns
Test data:        1000 rows, 7 columns
Test hidden data: 1000 rows, 8 columns

Training sentiment distribution:


sentiment
Positive    3749
Neutral      158
Negative      93
Name: count, dtype: int64

Missing values in training data:
name                  0
brand                 0
categories            0
primaryCategories     0
reviews.date          0
reviews.text          0
reviews.title        10
sentiment             0
dtype: int64


## Section 4: Text Preprocessing
Light cleaning for BERT (lowercase, remove URLs/HTML/special chars). Also a heavier version for ABSA that removes stopwords and lemmatizes.

In [4]:
# Build stopword set but KEEP negation words (critical for sentiment)
stop_words = set(stopwords.words('english'))
negation_words = {'not','no','nor','neither','never','none',
                  "don't","doesn't","didn't","won't","wouldn't",
                  "can't","cannot","couldn't","shouldn't","isn't",
                  "aren't","wasn't","weren't","hasn't","haven't"}
stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    """Light cleaning for BERT — keep structure, just remove noise."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # URLs
    text = re.sub(r'<.*?>', '', text)               # HTML tags
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)      # non-letters
    text = re.sub(r'\s+', ' ', text).strip()       # extra whitespace
    return text

def preprocess_for_analysis(text):
    """Heavier preprocessing for ABSA / word clouds (stopword removal + lemmatization)."""
    text = clean_text(text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)

# Apply light cleaning and combine title + text for richer BERT input
for df in [train_df, test_df, test_hidden_df]:
    df['clean_text']    = df['reviews.text'].apply(clean_text)
    df['reviews.title'] = df['reviews.title'].fillna('')
    df['combined_text'] = (df['reviews.title'].apply(clean_text) + ' ' + df['clean_text']).str.strip()

# Encode sentiment labels as integers
train_df['label']       = train_df['sentiment'].map(LABEL_MAP)
test_hidden_df['label'] = test_hidden_df['sentiment'].map(LABEL_MAP)

print(f"Sample cleaned text:")
print(f"  Original:  {train_df['reviews.text'].iloc[0][:100]}...")
print(f"  Cleaned:   {train_df['combined_text'].iloc[0][:100]}...")
print(f"\nPreprocessing complete for {len(train_df)} training samples.")


Sample cleaned text:
  Original:  Purchased on Black FridayPros - Great Price (even off sale)Very powerful and fast with quad core pro...
  Cleaned:   powerful tablet purchased on black fridaypros great price even off sale very powerful and fast with ...

Preprocessing complete for 4000 training samples.


## Section 5: Train / Validation Split
Stratified split so each sentiment class keeps the same ratio in both sets.

In [5]:
# Stratified split — keeps class proportions identical in train & val
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['combined_text'].values,
    train_df['label'].values,
    test_size=VALIDATION_SPLIT,
    random_state=RANDOM_SEED,
    stratify=train_df['label'].values,
)

print(f"Training samples:   {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print(f"\nTraining label distribution:")
for name, idx in LABEL_MAP.items():
    count = (train_labels == idx).sum()
    print(f"  {name}: {count} ({count/len(train_labels)*100:.1f}%)")


Training samples:   3400
Validation samples: 600

Training label distribution:
  Negative: 79 (2.3%)
  Neutral: 134 (3.9%)
  Positive: 3187 (93.7%)


## Section 6: Compute Class Weights
Handle class imbalance — minority classes (Negative) get higher weight so the model doesn't ignore them.

In [6]:
# Compute balanced class weights so minority classes get more attention
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_labels,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print("Class weights (higher = model pays more attention):")
for name, weight in zip(LABEL_NAMES, class_weights):
    print(f"  {name}: {weight:.4f}")
print(f"\nNegative reviews get ~{class_weights[0]/class_weights[2]:.1f}x more weight than Positive")


Class weights (higher = model pays more attention):
  Negative: 14.3460
  Neutral: 8.4577
  Positive: 0.3556

Negative reviews get ~40.3x more weight than Positive


## Section 7: Tokenization
Convert review text into token IDs that DistilBERT understands. Wrap everything in PyTorch DataLoaders.

In [7]:
# Load the DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    """PyTorch Dataset — tokenizes text on-the-fly and returns input_ids, attention_mask, labels."""
    def __init__(self, texts, labels=None):
        self.texts  = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            str(self.texts[idx]),
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Build datasets and loaders
train_dataset = SentimentDataset(train_texts, train_labels)
val_dataset   = SentimentDataset(val_texts,   val_labels)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# Quick sanity check — show one tokenized sample
sample = tokenizer(train_texts[0][:100], max_length=MAX_LENGTH, padding='max_length', truncation=True)
print(f"Sample tokenization:")
print(f"  Text: '{train_texts[0][:60]}...'")
print(f"  Token IDs (first 20): {sample['input_ids'][:20]}")
print(f"  Attention Mask (first 20): {sample['attention_mask'][:20]}")
print(f"\nDatasets created: {len(train_dataset)} train, {len(val_dataset)} validation")


Sample tokenization:
  Text: 'echo plus is fantastic alexa rocks the echo plus is super ea...'
  Token IDs (first 20): [101, 9052, 4606, 2003, 10392, 24969, 5749, 1996, 9052, 4606, 2003, 3565, 3733, 2000, 2275, 2039, 1045, 2573, 2307, 2293]
  Attention Mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Datasets created: 3400 train, 600 validation


## Section 8: Build & Train the Model
Load pre-trained DistilBERT, add a 3-class classification head, and fine-tune for 4 epochs with weighted cross-entropy loss.

In [8]:
# Load DistilBERT with a 3-class classification head on top
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(DEVICE)

# Optimizer with weight decay, plus a linear warmup scheduler
optimizer   = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

# Weighted loss so minority classes aren't ignored
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

# Track metrics across epochs
history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_accuracy': []}

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Epochs: {EPOCHS}, Batch Size: {BATCH_SIZE}, LR: {LEARNING_RATE}")
print(f"Total training steps: {total_steps}\n")

for epoch in range(EPOCHS):
    # ---- Training ----
    model.train()
    total_train_loss = 0
    train_steps = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()
        train_steps += 1

        if (batch_idx + 1) % 50 == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / train_steps

    # ---- Validation ----
    model.eval()
    total_val_loss = 0
    val_steps = 0
    all_preds, all_true = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = loss_fn(outputs.logits, labels)
            total_val_loss += loss.item()
            val_steps += 1

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / val_steps
    val_f1  = f1_score(all_true, all_preds, average='weighted')
    val_acc = accuracy_score(all_true, all_preds)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_f1'].append(val_f1)
    history['val_accuracy'].append(val_acc)

    print(f"\n  Epoch {epoch+1}/{EPOCHS} Summary:")
    print(f"    Train Loss: {avg_train_loss:.4f}")
    print(f"    Val Loss:   {avg_val_loss:.4f}")
    print(f"    Val F1:     {val_f1:.4f}")
    print(f"    Val Acc:    {val_acc:.4f}\n")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert-base-uncased
Parameters: 66,955,779
Epochs: 4, Batch Size: 16, LR: 2e-05
Total training steps: 852



  Epoch 1/4 | Batch 50/213 | Loss: 1.1091


  Epoch 1/4 | Batch 100/213 | Loss: 0.6655


...

  Epoch 4/4 | Batch 50/213 | Loss: 0.2520


  Epoch 4/4 | Batch 100/213 | Loss: 0.0267


  Epoch 4/4 | Batch 150/213 | Loss: 0.0098


  Epoch 4/4 | Batch 200/213 | Loss: 0.4666



  Epoch 4/4 Summary:
    Train Loss: 0.3955
    Val Loss:   0.7478
    Val F1:     0.9578
    Val Acc:    0.9600



## Section 9: Evaluation on Validation Set
Print a full classification report and save the confusion matrix as a PNG.

In [9]:
# Detailed per-class metrics on the validation set
print("Classification Report (Validation Set):")
print(classification_report(all_true, all_preds, target_names=LABEL_NAMES))

val_f1_final = f1_score(all_true, all_preds, average='weighted')
print(f"*** Weighted F1-Score: {val_f1_final:.4f} ***")
if val_f1_final > 0.85:
    print("✅ TARGET MET: F1-Score > 85%")
else:
    print("⚠️  F1-Score below 85% target — consider more epochs or tuning")

# Confusion matrix heatmap
cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title('Confusion Matrix — Validation Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_confusion_matrix_validation.png'), dpi=150)
plt.show()
print("Saved to outputs/01_confusion_matrix_validation.png")


Classification Report (Validation Set):
              precision    recall  f1-score   support

    Negative       0.69      0.79      0.73        14
     Neutral       0.65      0.46      0.54        24
    Positive       0.98      0.99      0.98       562

    accuracy                           0.96       600
   macro avg       0.77      0.74      0.75       600
weighted avg       0.96      0.96      0.96       600

*** Weighted F1-Score: 0.9578 ***
✅ TARGET MET: F1-Score > 85%


Saved to outputs/01_confusion_matrix_validation.png


## Section 10: Predict on Test Set & Evaluate
Run the trained model on the hidden test set, print metrics, save confusion matrix and predictions CSV.

In [10]:
# Evaluate on the hidden test set (has ground-truth labels)
test_dataset = SentimentDataset(test_hidden_df['combined_text'].values, test_hidden_df['label'].values)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
test_preds, test_true = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_true.extend(labels.cpu().numpy())

print("Classification Report (Test Set):")
print(classification_report(test_true, test_preds, target_names=LABEL_NAMES))

test_f1  = f1_score(test_true, test_preds, average='weighted')
test_acc = accuracy_score(test_true, test_preds)
print(f"*** Test Weighted F1-Score: {test_f1:.4f} ***")
print(f"*** Test Accuracy:          {test_acc:.4f} ***")

# Confusion matrix for test set
cm_test = confusion_matrix(test_true, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title('Confusion Matrix — Test Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_confusion_matrix_test.png'), dpi=150)
plt.show()
print("Saved to outputs/02_confusion_matrix_test.png")

# Also predict on the unlabelled test_data.csv and save to CSV
test_nolabel_dataset = SentimentDataset(test_df['combined_text'].values)
test_nolabel_loader  = DataLoader(test_nolabel_dataset, batch_size=BATCH_SIZE, shuffle=False)

final_preds = []
with torch.no_grad():
    for batch in test_nolabel_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        final_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())

test_df_output = test_df.copy()
test_df_output['predicted_sentiment'] = [LABEL_NAMES[p] for p in final_preds]
test_df_output[['name','reviews.text','reviews.title','predicted_sentiment']].to_csv(
    os.path.join(OUTPUT_DIR, 'test_predictions.csv'), index=False
)
print("Predictions saved to outputs/test_predictions.csv")


Classification Report (Test Set):
              precision    recall  f1-score   support

    Negative       0.68      0.71      0.69        24
     Neutral       0.54      0.38      0.45        39
    Positive       0.97      0.98      0.98       937

    accuracy                           0.95      1000
   macro avg       0.73      0.69      0.71      1000
weighted avg       0.95      0.95      0.95      1000

*** Test Weighted F1-Score: 0.9512 ***
*** Test Accuracy:          0.9540 ***
Saved to outputs/02_confusion_matrix_test.png


Predictions saved to outputs/test_predictions.csv


## Section 11: Training History Visualization
Plot loss, F1-score, and accuracy across epochs to see how training progressed.

In [11]:
# Visualize how loss, F1, and accuracy evolved over the 4 epochs
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(range(1, EPOCHS+1), history['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(range(1, EPOCHS+1), history['val_loss'],   'r-o', label='Val Loss')
axes[0].set_title('Training & Validation Loss');  axes[0].set_xlabel('Epoch');  axes[0].set_ylabel('Loss')
axes[0].legend();  axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, EPOCHS+1), history['val_f1'], 'g-o', label='Val F1-Score')
axes[1].axhline(y=0.85, color='r', linestyle='--', label='Target (0.85)')
axes[1].set_title('Validation F1-Score per Epoch');  axes[1].set_xlabel('Epoch');  axes[1].set_ylabel('F1-Score')
axes[1].legend();  axes[1].grid(True, alpha=0.3)

axes[2].plot(range(1, EPOCHS+1), history['val_accuracy'], 'm-o', label='Val Accuracy')
axes[2].set_title('Validation Accuracy per Epoch');  axes[2].set_xlabel('Epoch');  axes[2].set_ylabel('Accuracy')
axes[2].legend();  axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_training_history.png'), dpi=150)
plt.show()
print("Saved to outputs/03_training_history.png")


Saved to outputs/03_training_history.png


## Section 12: Aspect-Based Sentiment Analysis (ABSA)
Identify which product aspects (price, battery, screen, etc.) reviewers talk about, and what sentiment each aspect gets.

In [12]:
# Keyword dictionaries for each product aspect
ASPECT_KEYWORDS = {
    'Price/Value':       ['price','cost','expensive','cheap','value','money','worth','deal','affordable','budget','sale','bargain'],
    'Screen/Display':    ['screen','display','resolution','bright','hd','visual','color','picture','pixel','view'],
    'Battery/Power':     ['battery','charge','charging','power','last','outlet','plug','cord','usb'],
    'Sound/Audio':       ['sound','speaker','audio','volume','music','loud','bass','hear','listen','noise'],
    'Speed/Performance': ['speed','fast','slow','lag','performance','quick','responsive','processor','ram','memory'],
    'Build/Design':      ['build','quality','durable','sturdy','design','weight','light','heavy','size','compact','portable'],
    'Ease of Use':       ['easy','simple','intuitive','user-friendly','setup','navigate','interface','learn','beginner','convenient'],
    'Apps/Software':     ['app','apps','software','store','download','install','update','google','play','alexa','skill'],
    'Camera':            ['camera','photo','picture','video','record','selfie'],
    'Kids/Family':       ['kid','kids','child','children','son','daughter','grandkid','family','parent','parental','toddler'],
}

def extract_aspects(text):
    """Return list of aspects mentioned in a review."""
    text_lower = text.lower()
    return [asp for asp, kws in ASPECT_KEYWORDS.items() if any(kw in text_lower for kw in kws)]

# Tag every review with its aspects
train_df['aspects'] = train_df['reviews.text'].apply(extract_aspects)

# Count how often each aspect appears
all_aspects   = [a for aspects in train_df['aspects'] for a in aspects]
aspect_counts = Counter(all_aspects)

print("Aspect Frequency in Training Data:")
for aspect, count in aspect_counts.most_common():
    print(f"  {aspect}: {count} reviews ({count/len(train_df)*100:.1f}%)")

# Sentiment breakdown per aspect
print("\n--- Sentiment Breakdown by Aspect ---")
aspect_sentiment_data = []
for aspect in ASPECT_KEYWORDS:
    mask   = train_df['aspects'].apply(lambda x: aspect in x)
    subset = train_df[mask]
    if len(subset) > 0:
        dist = subset['sentiment'].value_counts(normalize=True) * 100
        pos, neu, neg = dist.get('Positive',0), dist.get('Neutral',0), dist.get('Negative',0)
        print(f"  {aspect:20s} | Pos: {pos:5.1f}% | Neu: {neu:5.1f}% | Neg: {neg:5.1f}% | Total: {len(subset)}")
        aspect_sentiment_data.append({'Aspect': aspect, 'Positive': pos, 'Neutral': neu, 'Negative': neg, 'Total Reviews': len(subset)})

aspect_df = pd.DataFrame(aspect_sentiment_data)

# Heatmap of sentiment % by aspect
plt.figure(figsize=(12, 6))
heatmap_data = aspect_df.set_index('Aspect')[['Positive','Neutral','Negative']]
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn', center=50,
            linewidths=0.5, cbar_kws={'label': 'Percentage (%)'})
plt.title('Aspect-Based Sentiment Analysis — Sentiment % by Aspect')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_absa_heatmap.png'), dpi=150)
plt.show()
print("Saved to outputs/04_absa_heatmap.png")

# Bar chart of aspect frequency
plt.figure(figsize=(12, 5))
aspects_sorted = sorted(aspect_counts.items(), key=lambda x: x[1], reverse=True)
names  = [a[0] for a in aspects_sorted]
counts_list = [a[1] for a in aspects_sorted]
bars = plt.barh(names[::-1], counts_list[::-1], color=sns.color_palette('viridis', len(names)))
plt.xlabel('Number of Reviews Mentioning Aspect')
plt.title('How Often Each Aspect is Discussed in Reviews')
for bar, c in zip(bars, counts_list[::-1]):
    plt.text(bar.get_width()+5, bar.get_y()+bar.get_height()/2, str(c), va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_aspect_frequency.png'), dpi=150)
plt.show()
print("Saved to outputs/05_aspect_frequency.png")


Aspect Frequency in Training Data:
  Apps/Software: 1293 reviews (32.3%)
  Kids/Family: 1199 reviews (30.0%)
  Ease of Use: 849 reviews (21.2%)
  Build/Design: 756 reviews (18.9%)
  Price/Value: 719 reviews (18.0%)
  Screen/Display: 689 reviews (17.2%)
  Sound/Audio: 609 reviews (15.2%)
  Battery/Power: 382 reviews (9.6%)
  Camera: 371 reviews (9.3%)
  Speed/Performance: 340 reviews (8.5%)

--- Sentiment Breakdown by Aspect ---
  Price/Value          | Pos:  92.2% | Neu:   5.1% | Neg:   2.6% | Total: 719
  Screen/Display       | Pos:  92.7% | Neu:   4.9% | Neg:   2.3% | Total: 689
  Battery/Power        | Pos:  88.0% | Neu:   6.5% | Neg:   5.5% | Total: 382
  Sound/Audio          | Pos:  94.7% | Neu:   2.5% | Neg:   2.8% | Total: 609
  Speed/Performance    | Pos:  89.1% | Neu:   7.4% | Neg:   3.5% | Total: 340
  Build/Design         | Pos:  94.3% | Neu:   3.3% | Neg:   2.4% | Total: 756
  Ease of Use          | Pos:  97.4% | Neu:   2.2% | Neg:   0.4% | Total: 849
  Apps/Software       

Saved to outputs/04_absa_heatmap.png


Saved to outputs/05_aspect_frequency.png


## Section 13: Category-Based Product Comparison
Group products into categories (Echo, Fire tablets, Kindle, etc.) and compare sentiment across them.

In [13]:
# Map product names to high-level categories
def categorize_product(name):
    n = name.lower()
    if 'echo show' in n:   return 'Echo Show'
    if 'echo plus' in n:   return 'Echo Plus'
    if 'tap' in n:         return 'Amazon Tap'
    if 'fire kids' in n:   return 'Fire Kids Tablet'
    if 'fire hd 10' in n:  return 'Fire HD 10'
    if 'fire hd 8' in n or 'fire hd8' in n: return 'Fire HD 8'
    if 'fire' in n and 'tablet' in n:        return 'Fire 7 Tablet'
    if 'kindle fire' in n: return 'Kindle Fire'
    if 'oasis' in n:       return 'Kindle Oasis'
    if 'voyage' in n:      return 'Kindle Voyage'
    if 'kindle' in n:      return 'Kindle E-reader'
    if 'fire tv' in n:     return 'Fire TV'
    if 'charger' in n or 'power' in n: return 'Accessories'
    return 'Other'

def get_product_type(cat):
    if cat in ['Echo Show','Echo Plus','Amazon Tap']:                          return 'Smart Speakers'
    if cat in ['Fire HD 8','Fire HD 10','Fire 7 Tablet','Fire Kids Tablet','Kindle Fire']: return 'Tablets'
    if cat in ['Kindle E-reader','Kindle Voyage','Kindle Oasis']:              return 'E-Readers'
    return 'Other'

train_df['product_category'] = train_df['name'].apply(categorize_product)
train_df['product_type']     = train_df['product_category'].apply(get_product_type)

# Sentiment by product category
print("--- Sentiment by Product Category ---")
cat_sentiment = train_df.groupby('product_category')['sentiment'].value_counts(normalize=True).unstack(fill_value=0) * 100
cat_counts    = train_df['product_category'].value_counts()
cat_sentiment = cat_sentiment.loc[cat_counts.index]

for cat in cat_sentiment.index:
    pos = cat_sentiment.loc[cat].get('Positive',0)
    neu = cat_sentiment.loc[cat].get('Neutral',0)
    neg = cat_sentiment.loc[cat].get('Negative',0)
    print(f"  {cat:20s} | Pos: {pos:5.1f}% | Neu: {neu:5.1f}% | Neg: {neg:5.1f}% | Reviews: {cat_counts[cat]}")

# Sentiment by broader product type
print("\n--- Sentiment by Product Type ---")
type_sentiment = train_df.groupby('product_type')['sentiment'].value_counts(normalize=True).unstack(fill_value=0) * 100
print(type_sentiment.round(1))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top_cats     = cat_counts[cat_counts >= 20].index
cat_plot_data = cat_sentiment.loc[top_cats]
if 'Positive' in cat_plot_data.columns:
    cat_plot_data[['Positive','Neutral','Negative']].plot(kind='barh', stacked=True, ax=axes[0], color=['#2ecc71','#f39c12','#e74c3c'])
axes[0].set_title('Sentiment Distribution by Product');  axes[0].set_xlabel('Percentage (%)');  axes[0].legend(title='Sentiment')

type_sentiment[['Positive','Neutral','Negative']].plot(kind='bar', ax=axes[1], color=['#2ecc71','#f39c12','#e74c3c'])
axes[1].set_title('Sentiment by Product Type');  axes[1].set_ylabel('Percentage (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right');  axes[1].legend(title='Sentiment')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_category_comparison.png'), dpi=150)
plt.show()
print("Saved to outputs/06_category_comparison.png")


--- Sentiment by Product Category ---
  Fire HD 8            | Pos:  94.9% | Neu:   3.3% | Neg:   1.8% | Reviews: 845
  Fire 7 Tablet        | Pos:  91.3% | Neu:   5.2% | Neg:   3.5% | Reviews: 766
  Echo Show            | Pos:  94.5% | Neu:   3.6% | Neg:   1.9% | Reviews: 676
  Fire Kids Tablet     | Pos:  92.6% | Neu:   5.0% | Neg:   2.4% | Reviews: 621
  Echo Plus            | Pos:  97.1% | Neu:   1.7% | Neg:   1.2% | Reviews: 483
  Kindle E-reader      | Pos:  91.0% | Neu:   6.2% | Neg:   2.8% | Reviews: 211
  Amazon Tap           | Pos:  91.0% | Neu:   4.5% | Neg:   4.5% | Reviews: 177
  Fire HD 10           | Pos:  95.5% | Neu:   3.4% | Neg:   1.1% | Reviews: 89
  Kindle Voyage        | Pos:  95.2% | Neu:   3.6% | Neg:   1.2% | Reviews: 84
  Kindle Oasis         | Pos:  97.8% | Neu:   0.0% | Neg:   2.2% | Reviews: 46
  Fire TV              | Pos: 100.0% | Neu:   0.0% | Neg:   0.0% | Reviews: 2

--- Sentiment by Product Type ---
sentiment       Negative  Neutral  Positive
product_

Saved to outputs/06_category_comparison.png


## Section 14: Overall Sentiment Visualizations
Pie chart of sentiment distribution and histogram of review lengths by sentiment class.

In [14]:
# Pie chart + review-length histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71', '#f39c12', '#e74c3c']

sentiment_counts = train_df['sentiment'].value_counts()
axes[0].pie(sentiment_counts.values, labels=sentiment_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90, explode=(0, 0.05, 0.1))
axes[0].set_title('Overall Sentiment Distribution (Training Data)')

train_df['text_length'] = train_df['reviews.text'].str.len()
for sent, color in zip(['Positive','Neutral','Negative'], colors):
    subset = train_df[train_df['sentiment'] == sent]['text_length']
    axes[1].hist(subset, bins=50, alpha=0.6, label=sent, color=color)
axes[1].set_title('Review Length Distribution by Sentiment')
axes[1].set_xlabel('Review Length (characters)');  axes[1].set_ylabel('Count')
axes[1].legend();  axes[1].set_xlim(0, 1000)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_sentiment_overview.png'), dpi=150)
plt.show()
print("Saved to outputs/07_sentiment_overview.png")


Saved to outputs/07_sentiment_overview.png


## Section 15: Word Clouds
Generate word clouds for Positive, Neutral, and Negative reviews to see the most common words in each class.

In [15]:
# Word clouds per sentiment class
try:
    from wordcloud import WordCloud

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for idx, (sent, cmap) in enumerate([('Positive','Greens'), ('Neutral','Oranges'), ('Negative','Reds')]):
        text = ' '.join(train_df[train_df['sentiment'] == sent]['reviews.text'].apply(preprocess_for_analysis))
        wc = WordCloud(width=600, height=300, background_color='white', colormap=cmap, max_words=80).generate(text)
        axes[idx].imshow(wc, interpolation='bilinear')
        axes[idx].set_title(f'{sent} Reviews — Word Cloud')
        axes[idx].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '08_word_clouds.png'), dpi=150)
    plt.show()
    print("Saved to outputs/08_word_clouds.png")
except Exception as e:
    print(f"Word cloud generation skipped: {e}")


Saved to outputs/08_word_clouds.png


## Section 16: Aspect × Product Type Heatmap
Cross-reference aspects with product types to see which aspects get the most positive sentiment per product category.

In [16]:
# Build a matrix: for each (product_type, aspect) pair, compute % positive sentiment
product_types = ['Tablets', 'Smart Speakers', 'E-Readers']
aspect_product_data = []

for ptype in product_types:
    subset = train_df[train_df['product_type'] == ptype]
    for aspect in ASPECT_KEYWORDS:
        mask = subset['aspects'].apply(lambda x: aspect in x)
        asp_reviews = subset[mask]
        pos_pct = (asp_reviews['sentiment'] == 'Positive').mean() * 100 if len(asp_reviews) > 0 else np.nan
        aspect_product_data.append({'Product Type': ptype, 'Aspect': aspect, 'Positive %': pos_pct})

ap_df    = pd.DataFrame(aspect_product_data)
ap_pivot = ap_df.pivot(index='Aspect', columns='Product Type', values='Positive %')

plt.figure(figsize=(10, 7))
sns.heatmap(ap_pivot, annot=True, fmt='.0f', cmap='RdYlGn', center=90,
            linewidths=0.5, cbar_kws={'label': 'Positive Sentiment %'}, vmin=70, vmax=100)
plt.title('Positive Sentiment % — Aspect × Product Type')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '09_aspect_product_heatmap.png'), dpi=150)
plt.show()
print("Saved to outputs/09_aspect_product_heatmap.png")


Saved to outputs/09_aspect_product_heatmap.png


## Section 17: Save Results Summary
Dump all key metrics and training history into a JSON file for easy reference.

In [17]:
# Collect all key results into a single JSON file
results_summary = {
    'model': MODEL_NAME,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'max_length': MAX_LENGTH,
    'validation_f1_score':  round(val_f1_final, 4),
    'validation_accuracy':  round(history['val_accuracy'][-1], 4),
    'test_f1_score':        round(test_f1, 4),
    'test_accuracy':        round(test_acc, 4),
    'training_samples':     len(train_texts),
    'validation_samples':   len(val_texts),
    'test_samples':         len(test_hidden_df),
    'class_weights':        {name: round(w, 4) for name, w in zip(LABEL_NAMES, class_weights)},
    'training_history':     history,
}

with open(os.path.join(OUTPUT_DIR, 'results_summary.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"Results summary saved to outputs/results_summary.json")
print(f"\n{'='*60}")
print(f"🎉 ALL DONE! PIPELINE COMPLETE")
print(f"{'='*60}")
print(f"\n  Validation F1-Score: {val_f1_final:.4f}")
print(f"  Test F1-Score:       {test_f1:.4f}")
print(f"  Test Accuracy:       {test_acc:.4f}")
print(f"\n  Output files in: {OUTPUT_DIR}")


Results summary saved to outputs/results_summary.json

🎉 ALL DONE! PIPELINE COMPLETE

  Validation F1-Score: 0.9578
  Test F1-Score:       0.9512
  Test Accuracy:       0.9540

  Output files in: /Users/remk/Documents/IIT Madras- Capstone project/Ecommerce-Sentiment-Analysis/personal_update/outputs
